In [7]:
import json
import os
import sys
import time

def unpack_sourcemap(map_path: str, output_dir: str):
    print(f"[*] 소스맵 파일 읽는 중: {map_path}")
    start = time.time()

    # 파일 크기 확인
    file_size = os.path.getsize(map_path)
    print(f"[*] 파일 크기: {file_size / 1024 / 1024:.1f} MB")

    # JSON 파싱
    print("[*] JSON 파싱 중...")
    try:
        with open(map_path, 'r', encoding='utf-8') as f:
            source_map = json.load(f)
        print(f"[+] 파싱 완료 ({time.time() - start:.2f}s)")
    except json.JSONDecodeError as e:
        print(f"[-] JSON 파싱 실패: {e}")
        sys.exit(1)
    except FileNotFoundError:
        print(f"[-] 파일을 찾을 수 없음: {map_path}")
        sys.exit(1)

    # 기본 정보 출력
    version = source_map.get('version', 'unknown')
    sources = source_map.get('sources', [])
    contents = source_map.get('sourcesContent', [])
    names = source_map.get('names', [])

    print(f"\n[*] 소스맵 정보")
    print(f"    버전        : {version}")
    print(f"    소스 파일 수 : {len(sources)}개")
    print(f"    names 수    : {len(names)}개")
    print(f"    sourceRoot  : {source_map.get('sourceRoot', '없음')}")
    print(f"    file        : {source_map.get('file', '없음')}")
    print(f"    sourcesContent 포함: {'yes' if contents else 'no'}")

    if not contents:
        print("[-] sourcesContent가 없어 소스 복원 불가")
        sys.exit(1)

    # 통계
    total = len(sources)
    skipped = 0
    success = 0
    errors = []

    print(f"\n[*] 파일 추출 시작 → {output_dir}/")
    print("-" * 50)

    for i, (file_path, content) in enumerate(zip(sources, contents)):
        progress = (i + 1) / total * 100
        print(f"\r[{progress:5.1f}%] ({i+1}/{total}) {file_path[:50]}", end='', flush=True)

        # null 콘텐츠 스킵
        if not content:
            skipped += 1
            continue

        try:
            full_path = os.path.join(output_dir, file_path.lstrip('/'))
            os.makedirs(os.path.dirname(full_path), exist_ok=True)

            with open(full_path, 'w', encoding='utf-8') as f:
                f.write(content)
            success += 1

        except Exception as e:
            errors.append((file_path, str(e)))

    elapsed = time.time() - start
    print(f"\n{'-' * 50}")
    print(f"\n[+] 완료! ({elapsed:.2f}s)")
    print(f"    성공   : {success}개")
    print(f"    스킵   : {skipped}개 (sourcesContent null)")
    print(f"    실패   : {len(errors)}개")

    if errors:
        print("\n[-] 실패 목록:")
        for path, err in errors:
            print(f"    {path}: {err}")


map_file = "/home/asus/paper/claude-code-2.1.88/package/cli.js.map"
out_dir  = "/home/asus/paper/claude-code-2.1.88/source/"
unpack_sourcemap(map_file, out_dir)

[*] 소스맵 파일 읽는 중: /home/asus/paper/claude-code-2.1.88/package/cli.js.map
[*] 파일 크기: 57.0 MB
[*] JSON 파싱 중...
[+] 파싱 완료 (1.04s)

[*] 소스맵 정보
    버전        : 3
    소스 파일 수 : 4756개
    names 수    : 0개
    sourceRoot  : 없음
    file        : 없음
    sourcesContent 포함: yes

[*] 파일 추출 시작 → /home/asus/paper/claude-code-2.1.88/source//
--------------------------------------------------
[100.0%] (4756/4756) ../src/entrypoints/cli.tsxde.tstopImportDialog.tsx
--------------------------------------------------

[+] 완료! (12.99s)
    성공   : 4756개
    스킵   : 0개 (sourcesContent null)
    실패   : 0개
